# Interactive Sphere Contact — Gaussian FEM Solver

Move the sphere around with sliders and see the FEM displacement results update in real time (~0.6s per solve).
- **Sliders**: control sphere X, Y, Z position, radius, deformation scale, and total force
- **Solve button**: click after adjusting sliders (or use continuous_update)
- The solver is pre-fitted once; only the solve step re-runs on each update

In [ ]:
import sys, time
import numpy as np
import torch
import h5py
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy.spatial import cKDTree

sys.path.insert(0, ".")
from gaussian_fem import GaussianFEM

DEVICE = "cuda"
E_MAT, NU_MAT = 113_800.0, 0.342
SAMPLE_PATH = "/data2/deepjeb/remeshed/sample_101_428.h5"

# ── Load mesh ──
with h5py.File(SAMPLE_PATH, "r") as f:
    coords_t = torch.from_numpy(f["coords"][:].astype(np.float32)).to(DEVICE)
    elems_t = torch.from_numpy(f["elements"][:].astype(np.int64)).to(DEVICE)
    rbe2_t = torch.from_numpy(f["rbe2"][:].astype(np.int64)).to(DEVICE)
    surfaces = f["surfaces"][:].astype(np.int64)

coords_np = coords_t.cpu().numpy()
rbe2_np = rbe2_t.cpu().numpy()
tris = surfaces
surf_node_ids = np.unique(surfaces)
surf_coords = coords_np[surf_node_ids]

# ── Precompute outward vertex normals and KDTree (one-time, <0.1s) ──
t0 = time.time()

v0 = coords_np[tris[:, 0]]
v1 = coords_np[tris[:, 1]]
v2 = coords_np[tris[:, 2]]
face_normals = np.cross(v1 - v0, v2 - v0)
vertex_normals = np.zeros_like(coords_np)
for i in range(3):
    np.add.at(vertex_normals, tris[:, i], face_normals)
vn_norms = np.linalg.norm(vertex_normals, axis=1, keepdims=True)
vertex_normals /= np.clip(vn_norms, 1e-12, None)

surf_normals = vertex_normals[surf_node_ids]
surf_tree = cKDTree(surf_coords)

print(f"Mesh: {coords_np.shape[0]} nodes, {tris.shape[0]} surface tris")
print(f"Fixed nodes: {len(rbe2_np)}")
print(f"Bounding box: {coords_np.min(0).round(1)} → {coords_np.max(0).round(1)}")
print(f"Surface normals + KDTree built in {time.time()-t0:.3f}s")

## Pre-fit the solver (one-time, ~3s)
This only runs once. After fitting, each solve takes ~0.6s.

In [5]:
solver = GaussianFEM(K=1024, broadening=1.5, enrichment="sgfem_linear",
                     fit_steps=300, device=DEVICE, seed=42)
t0 = time.time()
solver.fit(coords_t, elems_t, E_MAT, NU_MAT)
torch.cuda.synchronize()
print(f"Fit done in {time.time()-t0:.2f}s — P shape: {solver.P.shape}")

# ── Precompute stress B-matrix components (mesh-only, ~0.2s) ──
t0 = time.time()
elems_np = elems_t.cpu().numpy()
lam = E_MAT * NU_MAT / ((1 + NU_MAT) * (1 - 2 * NU_MAT))
mu = E_MAT / (2 * (1 + NU_MAT))

_v0 = coords_np[elems_np[:, 0]]
_J = np.stack([coords_np[elems_np[:, i]] - _v0 for i in range(1, 4)], axis=1)
_Jinv = np.linalg.inv(_J)
dN_precomp = np.transpose(_Jinv, (0, 2, 1))  # (M, 3, 3)

# Vertex averaging weights (precompute counts)
vm_count = np.zeros(len(coords_np), dtype=np.float64)
for i in range(4):
    np.add.at(vm_count, elems_np[:, i], 1.0)
vm_count = np.clip(vm_count, 1, None)

print(f"B-matrix precomputed in {time.time()-t0:.3f}s")


CONTACT_PEN_FRAC = 0.08  # sphere overlaps surface by 8% of radius

def snap_outside(center, radius):
    """If center is inside the mesh, snap it outside along the surface normal.
    Places the sphere so it just barely touches the surface (small overlap
    for contact detection)."""
    dist, idx = surf_tree.query(center)
    nearest_pt = surf_coords[idx]
    normal = surf_normals[idx]
    if np.dot(center - nearest_pt, normal) < 0:
        # Inside — push center out along normal, tangent minus small penetration
        center = nearest_pt + normal * (radius - radius * CONTACT_PEN_FRAC)
    return center


def compute_contact_and_solve(sphere_center, sphere_radius, total_force,
                              penetration_frac=0.15):
    sphere_center = snap_outside(sphere_center, sphere_radius)

    dist = np.linalg.norm(surf_coords - sphere_center, axis=1)
    pen_depth = sphere_radius * penetration_frac
    mask = (dist <= sphere_radius) & (dist >= sphere_radius - pen_depth)
    contact_ids = surf_node_ids[mask]

    N = coords_np.shape[0]
    f_ext_np = np.zeros((N, 3), dtype=np.float32)

    if len(contact_ids) > 0:
        radial = coords_np[contact_ids] - sphere_center
        norms = np.linalg.norm(radial, axis=1, keepdims=True)
        normals = radial / np.clip(norms, 1e-12, None)
        f_ext_np[contact_ids] = normals * (total_force / len(contact_ids))

    f_ext_t = torch.from_numpy(f_ext_np).to(DEVICE)
    u = solver.solve(f_ext_t, rbe2_t, n_iter=100)
    torch.cuda.synchronize()
    return u.cpu().numpy(), contact_ids, f_ext_np, sphere_center


def compute_von_mises_fast(disp_np):
    """Von Mises stress using precomputed B-matrix. Returns per-vertex values."""
    u_el = disp_np[elems_np]
    du = u_el[:, 1:] - u_el[:, 0:1]
    grad_u = np.einsum("mni,mnj->mij", du, dN_precomp)
    eps = 0.5 * (grad_u + np.transpose(grad_u, (0, 2, 1)))
    tr_eps = eps[:, 0, 0] + eps[:, 1, 1] + eps[:, 2, 2]
    sigma = 2 * mu * eps
    for i in range(3):
        sigma[:, i, i] += lam * tr_eps
    s = sigma - np.trace(sigma, axis1=1, axis2=2)[:, None, None] / 3 * np.eye(3)
    vm_elem = np.sqrt(1.5 * np.einsum("mij,mij->m", s, s))

    vm_vertex = np.zeros(len(coords_np), dtype=np.float64)
    for i in range(4):
        np.add.at(vm_vertex, elems_np[:, i], vm_elem)
    vm_vertex /= vm_count
    return vm_vertex

print("Ready for interactive solving.")

Fit done in 0.54s — P shape: torch.Size([39435, 4096])
B-matrix precomputed in 0.102s
Ready for interactive solving.


## Interactive Sphere Control
Move the sphere with sliders. Click **Solve** to recompute the FEM result, or toggle **Live** for auto-update on every slider change.

If you set the sphere inside the mesh, it will be automatically snapped to the nearest outside position (shown in the status bar).

In [6]:
# ── Slider setup ──
cmin, cmax = coords_np.min(0), coords_np.max(0)
margin = 15.0
style = {"description_width": "60px"}
layout = widgets.Layout(width="420px")

sx_slider = widgets.FloatSlider(value=6.6, min=cmin[0]-margin, max=cmax[0]+margin,
    step=0.5, description="X:", style=style, layout=layout, continuous_update=False)
sy_slider = widgets.FloatSlider(value=-71.4, min=cmin[1]-margin, max=cmax[1]+margin,
    step=0.5, description="Y:", style=style, layout=layout, continuous_update=False)
sz_slider = widgets.FloatSlider(value=55.0, min=cmin[2]-margin, max=cmax[2]+margin,
    step=0.5, description="Z:", style=style, layout=layout, continuous_update=False)
sr_slider = widgets.FloatSlider(value=5.0, min=2.0, max=20.0,
    step=0.5, description="Radius:", style=style, layout=layout, continuous_update=False)
force_slider = widgets.FloatSlider(value=35000, min=5000, max=200000,
    step=5000, description="Force:", style=style, layout=layout, continuous_update=False)
scale_slider = widgets.IntSlider(value=50, min=1, max=300,
    step=5, description="Disp.×:", style=style, layout=layout, continuous_update=False)
live_toggle = widgets.ToggleButton(value=False, description="Live",
    icon="refresh", layout=widgets.Layout(width="80px"))
solve_btn = widgets.Button(description="Solve", button_style="primary",
    icon="play", layout=widgets.Layout(width="100px"))

view_toggle = widgets.ToggleButtons(
    options=["Displacement", "Von Mises Stress"],
    value="Displacement",
    description="View:",
    style={"description_width": "50px", "button_width": "140px"},
)

status_label = widgets.HTML(value="<i>Adjust sliders then click Solve</i>")

# ── Sphere wireframe template ──
u_ang = np.linspace(0, 2*np.pi, 25)
v_ang = np.linspace(0, np.pi, 15)
_sx = np.outer(np.cos(u_ang), np.sin(v_ang))
_sy = np.outer(np.sin(u_ang), np.sin(v_ang))
_sz = np.outer(np.ones_like(u_ang), np.cos(v_ang))

# ═══════════════════════════════════════════════════════════
# Single figure with togglable coloring
# ═══════════════════════════════════════════════════════════
fig = go.FigureWidget()
fig.update_layout(
    width=950, height=650, margin=dict(l=0, r=0, t=40, b=0),
    scene=dict(aspectmode="data", xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
)
# 0: original mesh (ghost)
fig.add_trace(go.Mesh3d(
    x=coords_np[:, 0], y=coords_np[:, 1], z=coords_np[:, 2],
    i=tris[:, 0], j=tris[:, 1], k=tris[:, 2],
    color="lightgray", opacity=0.12, flatshading=True,
    name="Mesh (original)", hoverinfo="skip",
))
# 1: colored mesh (displacement OR stress)
fig.add_trace(go.Mesh3d(
    x=coords_np[:, 0], y=coords_np[:, 1], z=coords_np[:, 2],
    i=tris[:, 0], j=tris[:, 1], k=tris[:, 2],
    intensity=np.zeros(len(coords_np), dtype=np.float32),
    intensitymode="vertex",
    colorscale="Turbo",
    cmin=0, cmax=1,
    colorbar=dict(title="Disp. Mag.", len=0.6),
    flatshading=True, name="Result",
    hovertemplate="value=%{intensity:.4f}<extra></extra>",
))
# 2: sphere
fig.add_trace(go.Surface(
    x=_sx, y=_sy, z=_sz,
    colorscale=[[0, "rgba(255,165,0,0.06)"], [1, "rgba(255,165,0,0.06)"]],
    showscale=False, hoverinfo="skip", name="Sphere",
    contours=dict(
        x=dict(show=True, color="orange", width=2),
        y=dict(show=True, color="orange", width=2),
        z=dict(show=True, color="orange", width=2),
    ),
))
# 3: fixed nodes
fig.add_trace(go.Scatter3d(
    x=coords_np[rbe2_np, 0], y=coords_np[rbe2_np, 1], z=coords_np[rbe2_np, 2],
    mode="markers", marker=dict(size=3, color="blue", opacity=0.6),
    name="Fixed nodes",
))
# 4: contact nodes
fig.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode="markers", marker=dict(size=4, color="red", opacity=0.8),
    name="Contact nodes",
))
# 5: force cones
fig.add_trace(go.Cone(
    x=[0], y=[0], z=[0], u=[0], v=[0], w=[1],
    sizemode="absolute", sizeref=2, anchor="tail",
    colorscale=[[0, "crimson"], [1, "crimson"]], showscale=False,
    name="Forces",
))

# ── State: latest solve results ──
state = {"disp_mag": None, "vm_vertex": None, "deformed": None,
         "contact_ids": np.array([]), "sc_snapped": None, "sr": 5.0,
         "f_ext": None}


def update_view(_=None):
    """Switch the colored mesh between displacement and stress."""
    if state["disp_mag"] is None:
        return
    mode = view_toggle.value
    with fig.batch_update():
        if mode == "Displacement":
            dm = state["disp_mag"].astype(np.float32)
            fig.data[1].x = state["deformed"][:, 0]
            fig.data[1].y = state["deformed"][:, 1]
            fig.data[1].z = state["deformed"][:, 2]
            fig.data[1].intensity = dm
            fig.data[1].cmin = 0
            fig.data[1].cmax = float(dm.max()) if dm.max() > 0 else 1.0
            fig.data[1].colorscale = "Turbo"
            fig.data[1].colorbar = dict(title="Disp. Mag.", len=0.6)
            fig.data[1].hovertemplate = "disp=%{intensity:.5f}<extra></extra>"
            fig.update_layout(
                title=f"Displacement | max={dm.max():.5f}"
            )
        else:
            vm = state["vm_vertex"].astype(np.float32)
            vmax = float(np.percentile(vm[surf_node_ids], 99))
            vmax = max(vmax, 1e-6)
            fig.data[1].x = coords_np[:, 0]
            fig.data[1].y = coords_np[:, 1]
            fig.data[1].z = coords_np[:, 2]
            fig.data[1].intensity = np.clip(vm, 0, vmax)
            fig.data[1].cmin = 0
            fig.data[1].cmax = vmax
            fig.data[1].colorscale = "Hot"
            fig.data[1].colorbar = dict(title="Von Mises", len=0.6)
            fig.data[1].hovertemplate = "σ_vm=%{intensity:.1f}<extra></extra>"
            fig.update_layout(
                title=f"Von Mises Stress | max={vm.max():.0f} | 99th pct={vmax:.0f}"
            )

view_toggle.observe(lambda c: update_view(), names="value")


def do_solve(_=None):
    sc = np.array([sx_slider.value, sy_slider.value, sz_slider.value])
    sr = sr_slider.value
    total_f = force_slider.value
    scale = scale_slider.value

    status_label.value = "<b>Solving...</b>"
    t0 = time.time()
    disp, contact_ids, f_ext, sc_snapped = compute_contact_and_solve(sc, sr, total_f)
    dt_solve = time.time() - t0

    t0 = time.time()
    vm_vertex = compute_von_mises_fast(disp)
    dt_stress = time.time() - t0

    disp_mag = np.linalg.norm(disp, axis=1)
    deformed = coords_np + disp * scale
    snapped = not np.allclose(sc, sc_snapped, atol=0.01)

    # Save state
    state["disp_mag"] = disp_mag
    state["vm_vertex"] = vm_vertex
    state["deformed"] = deformed
    state["contact_ids"] = contact_ids
    state["sc_snapped"] = sc_snapped
    state["sr"] = sr
    state["f_ext"] = f_ext

    # Update shared traces (sphere, contacts, forces)
    with fig.batch_update():
        fig.data[2].x = sc_snapped[0] + sr * _sx
        fig.data[2].y = sc_snapped[1] + sr * _sy
        fig.data[2].z = sc_snapped[2] + sr * _sz

        if len(contact_ids) > 0:
            fig.data[4].x = coords_np[contact_ids, 0]
            fig.data[4].y = coords_np[contact_ids, 1]
            fig.data[4].z = coords_np[contact_ids, 2]
            fv = f_ext[contact_ids]
            fm = np.linalg.norm(fv, axis=1, keepdims=True)
            fd = fv / np.clip(fm, 1e-12, None)
            step = max(1, len(contact_ids) // 40)
            sel = np.arange(0, len(contact_ids), step)
            cp = coords_np[contact_ids[sel]]
            fig.data[5].x = cp[:, 0]
            fig.data[5].y = cp[:, 1]
            fig.data[5].z = cp[:, 2]
            fig.data[5].u = fd[sel, 0]
            fig.data[5].v = fd[sel, 1]
            fig.data[5].w = fd[sel, 2]
            fig.data[5].sizeref = sr * 0.35
        else:
            fig.data[4].x = [None]
            fig.data[4].y = [None]
            fig.data[4].z = [None]
            fig.data[5].x = [None]
            fig.data[5].y = [None]
            fig.data[5].z = [None]

    # Apply the current view mode (displacement or stress)
    update_view()

    snap_tag = " — <b style='color:orange'>snapped outside</b>" if snapped else ""
    status_label.value = (
        f"<b style='color:green'>Done</b> — {len(contact_ids)} contact, "
        f"max disp={disp_mag.max():.5f}, max σ={vm_vertex.max():.0f}, "
        f"solve={dt_solve:.2f}s + stress={dt_stress:.3f}s{snap_tag}"
    )

solve_btn.on_click(do_solve)

def on_slider_change(change):
    if live_toggle.value:
        do_solve()

for s in [sx_slider, sy_slider, sz_slider, sr_slider, force_slider, scale_slider]:
    s.observe(on_slider_change, names="value")

# ── Layout ──
sliders_box = widgets.VBox([
    widgets.HTML("<b>Sphere Position</b>"),
    sx_slider, sy_slider, sz_slider,
    widgets.HTML("<b>Parameters</b>"),
    sr_slider, force_slider, scale_slider,
    widgets.HBox([solve_btn, live_toggle, view_toggle]),
    status_label,
])

display(widgets.VBox([sliders_box, fig]))
do_solve()